[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/Lecture1_NN_torch.ipynb)

## Setup

Run the setup cell below once at the start of the session. A CPU runtime is sufficient; an optional GPU is detected automatically. The setup installs the required packages and downloads the dataset if needed. No repository clone or Google Drive mount is required.

Run the cells in order and complete the marked exercises. The placeholders deliberately raise `NotImplementedError` until you fill them in.

In [ ]:
# Install missing packages without upgrading the Colab environment.
#%pip -q install torch numpy matplotlib scikit-learn

from pathlib import Path
from urllib.request import urlretrieve
import torch

# Reuse a local copy, or download the dataset from the course repository.
data_path = Path("particle-events.npy")
local_data = Path("data/homogenous-16-particle-events-energy.npy")
if not data_path.exists():
    if local_data.exists():
        data_path = local_data
    else:
        data_url = (
            "https://raw.githubusercontent.com/rafaellopesdesa/ML_Mandi/main/"
            "activities/data/homogenous-16-particle-events-energy.npy"
        )
        temporary_path = data_path.with_suffix(".download")
        urlretrieve(data_url, temporary_path)
        temporary_path.replace(data_path)

print("PyTorch version:", torch.__version__)
print("Available device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Data file:", data_path)


# Neural Networks in `PyTorch`



We will use a dense neural network in `torch` to solve a simple regression problem using physics data.

## Learning Task
We will construct a dense neural network to predict the invariant mass of a particle from its energy, momentum, (and charge).

*Note that this task does not require machine learning. We choose a task with a known mapping to help us create, debug, and tune our first neural network.*

## Dataset
This dataset is a collection of simulated particle events from [Pythia](https://pythia.org/). The dataset is a 2D array where each row represents one event from an $e^{-} + p$ collision. This dataset is comprised _only_ of events where exactly 16 particles are produced from an electron-proton collision. Each particle contains $(p_x,p_y,p_z,E,q)$ information. Each event is therefore represented by 80 numbers.

**Advanced activity:** There are more interesting event-wise learning tasks using this dataset. Consider crafting your own learning task and target for this data.



## Computational Notes

If this is your first time in a Jupyter-like environment, please read the following carefully:

 - You are in an active kernel
 - Run each cell with `Shift + Enter`
 - You must execute the cells in the order that you want the code to run
 - `Runtime`$\rightarrow$`Change runtime type` allows you to select a GPU. A CPU is sufficient for this notebook; a TPU is not used here.


In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)


## Data loading

The setup cell downloads the approximately 23 MB dataset from this course's repository if a local copy is not available. No Google Drive mount is needed. The data originally come from the [NuclearTalent machine-learning course](https://github.com/NuclearTalent/MachineLearningECT/tree/master/doc/pub/Day6/data).


In [ ]:
# Load the NumPy array; each row is one event with 16 particles.
events = np.load(data_path, allow_pickle=False)
assert events.ndim == 2 and events.shape[1] == 16 * 5
print("Event array shape:", events.shape)


Each row currently represents an entire event. We want one particle per row.

Use `reshape` to obtain an array of shape `(number_of_particles, 5)`, then keep only $(p_x,p_y,p_z,E)$ as the four input features. Exclude charge for now: it is not needed in the invariant-mass formula.

We will initially use only **100 training particles**, selected after separating training, validation, and test events. Keep all particles at this step so that more training data remain available for tuning.


In [ ]:
n_part_per_event = 16
n_feat_per_part = 5  # px, py, pz, E, charge
evt_particles = events.reshape(len(events), n_part_per_event, n_feat_per_part)

# Exercise: reshape to one particle per row and retain the first four columns.
# Store the result in particles, with shape (len(events) * 16, 4).
raise NotImplementedError("Create the four-feature particle array.")


Our targets are the invariant masses of the particles. In units where $c=1$,
$$m^2 = E^2 - (p_x^2 + p_y^2 + p_z^2).$$

**Create a one-dimensional array `m` of masses, not mass squared.** Compute the targets with the original NumPy `float64` data, before converting the inputs to PyTorch `float32` tensors.

The stored energy and momentum have limited precision, so the subtraction can produce negative values near zero. Use `np.maximum(m_squared, 0)` before taking the square root. Clipping avoids `NaN`; it does not recover precision lost in the source data.


In [ ]:
# Exercise: calculate m_squared from particles, then take the clipped square root.
raise NotImplementedError("Calculate the invariant masses.")


Next, make a histogram of the target data to make sure that we are seeing masses of real particles. As this data has limited precision, this will not resolve electrons very well, but protons, pions, and massless particles should be clearly visible.

In [ ]:
plt.hist(m, bins=100)
plt.xlabel("mass (GeV)")
plt.ylabel("count")
plt.title("target distribution")
plt.show()


## Train/validation/test split and DataLoaders

First reserve 20% of the **events** for testing, then 20% of the remaining events for validation, using `train_test_split(..., random_state=42)` twice. This gives approximately 64%/16%/20% of events. Splitting events before selecting particles keeps correlated particles from one collision in the same partition.

Convert each event index to its 16 particle indices using the supplied helper. Keep the full index arrays as `train_pool`, `val_pool`, and `test_pool`. Then randomly select 100 training particles and 1000 particles each for validation and testing. These final particle counts are chosen for this small demonstration; they need not have the same proportions as the event split.

Create `X_train`, `X_val`, `X_test` and the corresponding `y_train`, `y_val`, `y_test`. Targets must have shape `(N, 1)` in PyTorch, matching the single output of the model. Use validation data for tuning and reserve test data for the final comparison.


In [ ]:
def particle_indices(event_indices):
    return (event_indices[:, None] * n_part_per_event
            + np.arange(n_part_per_event)).reshape(-1)

n_train = 100
n_val = 1000
n_test = 1000

# Exercise: split np.arange(len(events)) into train_events, val_events, test_events.
# Use particle_indices to create train_pool, val_pool, test_pool.
# Sample n_train/n_val/n_test indices without replacement using default_rng(42).
# Use those indices to select X_train/y_train, X_val/y_val, X_test/y_test.
raise NotImplementedError("Split the events and select the particle datasets.")

# Exercise: construct train_ds, val_ds, and test_ds with TensorDataset.
# Convert inputs to float32 and use .unsqueeze(1) for one-dimensional targets.
raise NotImplementedError("Create the three TensorDatasets.")

input_dim = X_train.shape[1]
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


I wrote a simple function for plotting loss curves, which will help us tune our model.

In [ ]:
def plot_learning_curve(history):
    epochs = np.arange(1, len(history["loss"]) + 1)
    plt.plot(epochs, history["loss"], label="training loss")
    plt.plot(epochs, history["val_loss"], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Mean squared error (GeV²)")
    plt.yscale("log")
    plt.legend()
    plt.show()


## Now we build our first model architecture!

Begin with four inputs, **one hidden layer with two ReLU nodes**, and one linear output for regression. This deliberately small network is a starting point for tuning.

Complete the final layer below. How many trainable parameters, including biases, does this network have?


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Exercise: complete nn.Sequential with a linear 2-to-1 output layer.
        raise NotImplementedError("Add the linear output layer.")
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2),
            nn.ReLU(),
            # Add the output layer here.
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
torch.manual_seed(42)
model = MLP(input_dim).to(device)
print(model)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)


## Optimizer, loss, and training loop

The functions below provide training and evaluation for one full pass through a DataLoader. Compare them: evaluation uses `model.eval()` and disables gradient tracking, and does not update the weights. The reported training loss averages mini-batches while parameters change; validation loss is evaluated with the parameters at the end of the epoch.


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_f):
    # Put the model in training mode
    model.train()
    running_loss = 0.0

    # Loop over all mini-batches from the DataLoader
    for xb, yb in loader:
        # Move both inputs (xb) and targets (yb) onto the same device (CPU or GPU)
        xb, yb = xb.to(device), yb.to(device)
        # Reset gradients from the previous step
        optimizer.zero_grad()
        # Forward pass: compute predictions for this batch
        preds = model(xb)
        # Compute the loss comparing predictions vs. true labels
        loss = loss_f(preds, yb)
        # Backward pass: compute gradients of all model parameters w.r.t. loss
        loss.backward()
        # Take one optimization step (update model weights using gradients)
        optimizer.step()
        # Accumulate the loss, scaled by batch size (for averaging later)
        running_loss += loss.item() * xb.size(0)
    # Return the average loss across the whole dataset
    return running_loss / len(loader.dataset)


# the following is for evaluating the model, e.g. for validation
# compare this with the training loop above and try to understand the differences
@torch.no_grad()
def evaluate(model, loader, loss_f):
    model.eval()
    running_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        loss = loss_f(preds, yb)
        running_loss += loss.item() * xb.size(0)
    return running_loss / len(loader.dataset)


In [ ]:
# Rerun this cell to start a fresh training run.
torch.manual_seed(42)
model = MLP(input_dim).to(device)
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
loss_f = nn.MSELoss()
history = {"loss": [], "val_loss": []}

epochs = 10
for epoch in range(1, epochs + 1):
    # Exercise: call train_one_epoch and evaluate to obtain the two losses.
    raise NotImplementedError("Complete the training and validation calls.")
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")


### Evaluate the model

Plot the loss curves and compare predictions with the test targets, both as histograms with shared bins and as predicted-versus-true masses. A matching histogram alone does not demonstrate accurate particle-by-particle predictions.

The first test evaluation is a baseline demonstration. Do not use it to choose hyperparameters: make subsequent choices with validation data, then compare the final model on the test set once.


In [ ]:
plot_learning_curve(history)

model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for xt, yt in test_loader:
        xt = xt.to(device)
        # Exercise: predict the masses for this batch and store them in preds.
        raise NotImplementedError("Predict with the trained model.")
        all_preds.append(preds.cpu().numpy())
        all_targets.append(yt.numpy())

test_preds = np.concatenate(all_preds).ravel()
test_true = np.concatenate(all_targets).ravel()
baseline_test_mse = np.mean((test_preds - test_true)**2)
print(f"Baseline test MSE: {baseline_test_mse:.6f} GeV²")

def plot_predictions(targets, predictions):
    low = min(targets.min(), predictions.min())
    high = max(targets.max(), predictions.max())
    bins = np.linspace(low, high, 51)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(targets, bins=bins, histtype="step", label="true")
    axes[0].hist(predictions, bins=bins, histtype="step", label="predicted")
    axes[0].set(xlabel="Mass (GeV)", ylabel="Particles")
    axes[0].legend()
    axes[1].scatter(targets, predictions, s=8, alpha=0.3)
    axes[1].plot([low, high], [low, high], "k--")
    axes[1].set(xlabel="True mass (GeV)", ylabel="Predicted mass (GeV)")
    fig.tight_layout()
    plt.show()

plot_predictions(test_true, test_preds)


## Tuning

The first architecture and training settings are deliberately simple. Use validation loss to explore:

1. Scaling the four input features. Fit the scaling on **training data only**, then apply it to validation and test data.
2. Adding charge. Is it needed for the exact mass formula? Could it still help a finite network on this particular dataset?
3. Using more training particles from `train_pool`, while keeping validation and test events separate.
4. Trying a different hidden activation, such as `Tanh` or `Sigmoid`.
5. Increasing the hidden-layer width or adding a hidden layer.
6. Changing batch size, learning rate, or optimizer. Smaller batches make noisier gradient estimates; they do not guarantee a global minimum. A large learning rate can cause oscillation or divergence, while a small one can make learning slow.
7. Training longer. If training loss falls while validation loss rises, consider early stopping, a smaller model, more data, weight decay, or dropout. Batch normalization may help optimization, but is not a general cure for overfitting.

Try one simple improvement and plot its training and validation curves. Only after choosing the final configuration, evaluate it on the test set.

Even after tuning, why is it hard for a generic neural network to predict very small masses from energy and momentum? Does the known formula leave an irreducible statistical uncertainty for the targets defined in this notebook?


In [ ]:
# Exercise: try a simple tuned model using the same event partitions.
# Fit any preprocessing on training data only.
# Train, plot the loss curves, and compare the final test predictions.
raise NotImplementedError("Try and evaluate a tuned model.")
